# Log Analytics System for a Web Server
**Big Data Essentials: 50% review demo (Google Colab)**

Team: **Kriti Singh** (RA2411028010081) · **Kushal Mohan** (RA2411028010082) · **Gargi Singh** (RA2411028010102)
Guide: **Dr. Senthil Kumar K.**

**Pipeline:** NASA web logs → **Hadoop HDFS** (store) → **Apache Spark** (clean / preprocess) → **Spark SQL** (reports + Z-score anomaly detection) → **Spark MLlib** (attack detection) → charts

| Member | Big data tool | Part |
|---|---|---|
| Kriti | **Hadoop HDFS** | Start HDFS, create the zones, load (ingest) the raw logs |
| Kushal | **Apache Spark (PySpark)** | ETL: raw text → clean typed table (Parquet, split by date) |
| Gargi | **Spark SQL** | 11 reports + Z-score anomaly detection |
| Team | **Spark MLlib** | Web-attack detection: Factorization Machine vs 7 models, confusion matrices |

> **Run:** `Runtime → Run all`. The first run takes about 5–8 minutes (it installs Java, Hadoop and Spark and downloads the data).
> On our laptops the same code runs on a Docker cluster with 2 DataNodes. Colab is one machine, so here HDFS has 1 NameNode + 1 DataNode.

## 0. Setup: Java, PySpark, Hadoop
Hadoop and Spark run on Java, so we install Java 11, PySpark 3.5.3 and Hadoop 3.3.6 (the same versions as our Docker image).

In [ ]:
%%bash
# Java 11 (Hadoop 3.3 runs on Java 8 or 11)
if [ ! -d /usr/lib/jvm/java-11-openjdk-amd64 ]; then
  apt-get update -qq > /dev/null 2>&1
  apt-get install -y -qq openjdk-11-jdk-headless > /dev/null 2>&1
fi
ls -d /usr/lib/jvm/java-11-openjdk-amd64 2>/dev/null && echo "Java 11 installed"

# Hadoop 3.3.6 (about 700 MB, 1-2 minutes)
WORK=/content
if [ ! -d $WORK/hadoop-3.3.6 ]; then
  for base in https://dlcdn.apache.org https://archive.apache.org/dist; do
    if curl -fsSL --retry 3 $base/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz -o $WORK/hadoop.tgz; then
      tar -xzf $WORK/hadoop.tgz -C $WORK && rm -f $WORK/hadoop.tgz
      break
    fi
  done
fi
if [ -d $WORK/hadoop-3.3.6 ]; then echo "Hadoop 3.3.6 ready"; else echo "Hadoop could not be downloaded"; fi

In [ ]:
# Python side of the setup: environment for Java, Hadoop and Spark (runs AFTER the install cell)
import os, sys, subprocess, shutil, time, glob

WORK = "/content"
os.makedirs(WORK, exist_ok=True)

def sh(cmd, check=False):
    """Run a shell command and print its output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
    return r.returncode == 0

J11 = "/usr/lib/jvm/java-11-openjdk-amd64"
if os.path.isdir(J11):
    os.environ["JAVA_HOME"] = J11
else:   # fall back to whatever java is installed
    os.environ["JAVA_HOME"] = os.path.dirname(os.path.dirname(os.path.realpath(shutil.which("java"))))
HADOOP_HOME = f"{WORK}/hadoop-3.3.6"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["HADOOP_CONF_DIR"] = f"{HADOOP_HOME}/etc/hadoop"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
os.environ.setdefault("USER", "root")
os.environ["PYSPARK_PYTHON"] = os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable   # Spark workers use this same Python

# PySpark 3.5.x (Colab usually has it already)
from importlib import metadata
try:
    ok = metadata.version("pyspark").startswith("3.5")
except metadata.PackageNotFoundError:
    ok = False
if not ok:
    sh(f"{sys.executable} -m pip install -q pyspark==3.5.3")

sh("java -version")
print("PySpark    ", metadata.version("pyspark"))
print("JAVA_HOME  ", os.environ["JAVA_HOME"])
print("HADOOP_HOME", HADOOP_HOME, "(found)" if os.path.isdir(HADOOP_HOME) else "(missing)")

## 1. Download the raw data (NASA-HTTP logs, July + August 1995)
Real access logs of NASA Kennedy Space Center's web server: about 37 MB compressed, 356 MB as text.

In [ ]:
DATA = f"{WORK}/data"
os.makedirs(DATA, exist_ok=True)
MIRROR = "https://raw.githubusercontent.com/greymd/NASA-HTTP/main"
for month in ["Jul95", "Aug95"]:
    local = f"{DATA}/access_log_{month}"
    if not os.path.exists(local):
        sh(f"curl -fsSL --retry 3 {MIRROR}/NASA_access_log_{month}.gz -o {local}.gz && gunzip -f {local}.gz", check=True)
sh(f"ls -lh {DATA}")
sh(f"wc -l {DATA}/access_log_*")

## 2. Kriti — Hadoop HDFS: start HDFS and load the raw logs
**HDFS** = Hadoop Distributed File System. The **NameNode** keeps the index (file names, blocks, locations); the **DataNode** stores the actual 128 MB blocks.

Same config as our Docker cluster (`conf/core-site.xml`, `conf/hdfs-site.xml`). Only 3 values change for Colab:

| Setting | Docker cluster | Colab (one machine) |
|---|---|---|
| `fs.defaultFS` | `hdfs://namenode:8020` | `hdfs://localhost:8020` |
| `dfs.replication` | 2 (2 DataNodes) | 1 (1 DataNode) |
| storage folders | `/hadoop/dfs/name`, `/hadoop/dfs/data` | `/content/hdfs/name`, `/content/hdfs/data` |

In [ ]:
%%bash
# Write core-site.xml and hdfs-site.xml (same settings as conf/ in our project)
CONF=/content/hadoop-3.3.6/etc/hadoop
if [ ! -d "$CONF" ]; then
  echo "Hadoop is missing: run the setup cell first"
  exit 0
fi

cat > "$CONF/core-site.xml" <<'XML'
<?xml version="1.0" encoding="UTF-8"?>
<configuration>
  <!-- Every component reaches HDFS through this one address. -->
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:8020</value>
  </property>
  <property>
    <name>hadoop.tmp.dir</name>
    <value>/content/hdfs/tmp</value>
  </property>
</configuration>
XML

cat > "$CONF/hdfs-site.xml" <<'XML'
<?xml version="1.0" encoding="UTF-8"?>
<configuration>
  <!-- 1 DataNode in Colab, so 1 copy (our Docker cluster: 2 DataNodes, replication 2) -->
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:///content/hdfs/name</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file:///content/hdfs/data</value>
  </property>
  <!-- same as our project: container hostnames don't reverse-resolve -->
  <property>
    <name>dfs.namenode.datanode.registration.ip-hostname-check</name>
    <value>false</value>
  </property>
  <property>
    <name>dfs.permissions.enabled</name>
    <value>false</value>
  </property>
</configuration>
XML

grep -q "^export JAVA_HOME=" "$CONF/hadoop-env.sh" || echo "export JAVA_HOME=$JAVA_HOME" >> "$CONF/hadoop-env.sh"
echo "== core-site.xml"; cat "$CONF/core-site.xml"
echo; echo "== hdfs-site.xml"; cat "$CONF/hdfs-site.xml"

In [ ]:
# Start HDFS: the same steps as docker/entrypoint.sh
#   format the NameNode only on the first run, start the NameNode, then the DataNode
HDFS_OK = False
if os.path.isfile(f"{HADOOP_HOME}/etc/hadoop/core-site.xml"):
    if not os.path.isdir(f"{WORK}/hdfs/name/current"):
        print("Formatting HDFS (first run only) ...")
        sh("hdfs namenode -format -force -nonInteractive > /dev/null 2>&1")
    running = subprocess.run("jps", shell=True, capture_output=True, text=True).stdout
    if " NameNode" not in running:
        subprocess.Popen("hdfs namenode", shell=True, start_new_session=True,
                         stdout=open(f"{WORK}/namenode.log", "w"), stderr=subprocess.STDOUT)
        time.sleep(8)
    if " DataNode" not in running:
        subprocess.Popen("hdfs datanode", shell=True, start_new_session=True,
                         stdout=open(f"{WORK}/datanode.log", "w"), stderr=subprocess.STDOUT)
    # wait until the NameNode leaves safe mode (read-only start-up state)
    for i in range(40):
        r = subprocess.run("hdfs dfsadmin -safemode get", shell=True, capture_output=True, text=True)
        if "OFF" in r.stdout:
            HDFS_OK = True
            break
        time.sleep(3)

if HDFS_OK:
    print("HDFS is ready\n")
    sh("jps")
    sh("hdfs dfsadmin -report | head -14")
else:
    print("HDFS did not start: the demo continues on local disk.")
    sh(f"tail -5 {WORK}/namenode.log 2>/dev/null")

**Ingest.** This is the `make ingest` target of our Makefile (`$(EXEC)` = `docker compose exec -T client`):
```make
ingest:          ## copy the raw logs from ./data into HDFS
	$(EXEC) hdfs dfs -mkdir -p /data/weblogs/raw /data/weblogs/curated /data/weblogs/reports /spark-logs
	$(EXEC) bash -c 'hdfs dfs -put -f /workspace/data/access_log_* /data/weblogs/raw/'
	$(EXEC) hdfs dfs -du -h /data/weblogs/raw
```
Line 65 creates the zones, **line 67 is the ingest line**, line 69 checks the result. The same commands run below. The raw zone is never edited.

In [ ]:
if HDFS_OK:
    BASE = "hdfs://localhost:8020/data/weblogs"
    # Makefile line 65: create the HDFS layout
    sh("hdfs dfs -mkdir -p /data/weblogs/raw /data/weblogs/curated /data/weblogs/reports")
    # Makefile line 67: THE INGEST LINE: copy the raw logs into HDFS
    sh(f"hdfs dfs -put -f {DATA}/access_log_* /data/weblogs/raw/", check=True)
    # Makefile line 69: check what landed
    sh("hdfs dfs -du -h /data/weblogs/raw")
else:
    BASE = f"file://{WORK}/weblogs"
    os.makedirs(f"{WORK}/weblogs/raw", exist_ok=True)
    for f in glob.glob(f"{DATA}/access_log_*"):
        dst = f"{WORK}/weblogs/raw/" + os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy(f, dst)

RAW, CURATED, REPORTS = f"{BASE}/raw", f"{BASE}/curated", f"{BASE}/reports"
print("RAW     =", RAW)
print("CURATED =", CURATED)
print("REPORTS =", REPORTS)

In [ ]:
# make fsck: every file is split into 128 MB blocks
if HDFS_OK:
    sh("hdfs fsck /data/weblogs/raw -files -blocks -locations | grep -E 'access_log|blk_|Total blocks|Average block replication'")

## 3. Kushal — Apache Spark: preprocessing (ETL)
**Spark** processes data in memory, split into partitions that run in parallel. Settings are the same as `conf/spark-defaults.conf`.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, LongType

spark = (SparkSession.builder
         .appName("weblog-etl")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.session.timeZone", "America/New_York")   # the logs are in NASA's local time (-0400)
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark

### 3.1 Before: the raw data (one long text line per request, no columns)

In [ ]:
raw = spark.read.text(RAW)
total = raw.count()
print(f"raw lines: {total:,}")
raw.show(5, truncate=False)
raw.printSchema()

### 3.2 Parse: cut every line into columns with regular expressions (`jobs/etl.py`)
Common Log Format: `host - - [timestamp] "METHOD /page PROTOCOL" status bytes`

In [ ]:
HOST   = r'^(\S+)'
TS     = r'\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2} [+\-]\d{4})\]'
REQ    = r'"(\S+)\s(\S+)\s*(\S*)"'
STATUS = r'\s(\d{3})\s'
BYTES  = r'\s(\d+|-)$'

parsed = (raw
    .select(
        F.col("value").alias("raw"),
        F.regexp_extract("value", HOST, 1).alias("host"),
        F.regexp_extract("value", TS, 1).alias("ts_str"),
        F.regexp_extract("value", REQ, 1).alias("method"),
        F.regexp_extract("value", REQ, 2).alias("endpoint"),
        F.regexp_extract("value", REQ, 3).alias("protocol"),
        F.regexp_extract("value", STATUS, 1).cast(IntegerType()).alias("status"),
        F.regexp_extract("value", BYTES, 1).alias("bytes_str"))
    # "-" in the bytes field means no body = 0 bytes
    .withColumn("bytes", F.when(F.col("bytes_str") == "-", F.lit(0))
                          .otherwise(F.col("bytes_str").cast(LongType())))
    .withColumn("ts", F.to_timestamp("ts_str", "dd/MMM/yyyy:HH:mm:ss Z"))
    .drop("ts_str", "bytes_str"))

### 3.3 Clean + derive: drop broken lines (kept as rejects), add date, hour, day of week, file extension

In [ ]:
bad = parsed.filter(F.col("ts").isNull() | F.col("status").isNull() | (F.col("host") == ""))
n_bad = bad.count()

clean = (parsed
    .filter(F.col("ts").isNotNull() & F.col("status").isNotNull() & (F.col("host") != ""))
    .drop("raw")
    .withColumn("date", F.to_date("ts"))
    .withColumn("hour", F.hour("ts"))
    .withColumn("dow",  F.date_format("ts", "EEEE"))
    .withColumn("ext",  F.lower(F.regexp_extract("endpoint", r"\.([A-Za-z0-9]{1,5})$", 1))))

print("rejected (malformed) lines:", n_bad)
bad.select("raw").show(truncate=False)

### 3.4 Write: Snappy Parquet, split into one folder per date → the **curated zone**

In [ ]:
if n_bad:
    bad.select("raw").limit(1000).write.mode("overwrite").text(CURATED + "_rejects")

(clean.write.mode("overwrite")
      .option("compression", "snappy")
      .partitionBy("date")
      .parquet(CURATED))

curated = spark.read.parquet(CURATED)
n_clean = curated.count()
print({"raw_lines": total, "clean_records": n_clean, "malformed": n_bad,
       "parse_rate_pct": round(100 * n_clean / total, 3)})

## 4. Preprocessing result: the raw text is now a table

In [ ]:
# After: 11 typed columns
curated.printSchema()

In [ ]:
curated.orderBy("ts").show(15)

In [ ]:
# Same request, before and after
line = raw.filter(F.col("value").startswith("199.72.81.55 - - [01/Jul/1995:00:00:01")).first()["value"]
print("BEFORE (raw text):\n ", line, "\n\nAFTER (the same request as one row of the table):")
curated.filter((F.col("host") == "199.72.81.55") & (F.col("date") == "1995-07-01")).orderBy("ts").limit(1).show(truncate=False)

In [ ]:
# The curated zone on disk: one folder per date
jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()
path = jvm.org.apache.hadoop.fs.Path(CURATED)
folders = sorted(s.getPath().getName() for s in path.getFileSystem(hconf).listStatus(path) if s.isDirectory())
print(len(folders), "date folders:", folders[:3], "...", folders[-2:])
if HDFS_OK:
    sh("hdfs dfs -du -s -h /data/weblogs/raw /data/weblogs/curated")

## 5. Gargi — Spark SQL: the 11 reports (`jobs/reports.py`)
The clean table is registered as a view called `access_logs`, then every question is plain SQL.

In [ ]:
curated.createOrReplaceTempView("access_logs")
curated.cache().count()

def report(name, sql, limit=None, n=10):
    df = spark.sql(sql)
    if limit:
        df = df.limit(limit)
    df.write.mode("overwrite").parquet(f"{REPORTS}/{name}")
    print(f"=== {name} ===")
    df.show(n, truncate=60)
    return df

In [ ]:
report("by_hour", """
    SELECT hour, COUNT(*) AS hits, COUNT(DISTINCT host) AS visitors,
           ROUND(SUM(bytes)/1024/1024/1024, 2) AS gb
    FROM access_logs GROUP BY hour ORDER BY hour""", n=24)

report("daily", """
    SELECT date, COUNT(*) AS hits, COUNT(DISTINCT host) AS visitors
    FROM access_logs GROUP BY date ORDER BY date""")

report("by_weekday", """
    SELECT dow, COUNT(*) AS hits FROM access_logs GROUP BY dow ORDER BY hits DESC""")

In [ ]:
report("top_urls", """
    SELECT endpoint, COUNT(*) AS hits FROM access_logs
    WHERE status = 200 AND ext NOT IN ('gif','jpg','jpeg','xbm','png','ico')
    GROUP BY endpoint ORDER BY hits DESC""", limit=300)

report("by_content_type", """
    SELECT CASE WHEN ext = '' THEN '(none)' ELSE ext END AS ext,
           COUNT(*) AS hits, ROUND(SUM(bytes)/1024/1024/1024, 2) AS gb
    FROM access_logs GROUP BY ext ORDER BY hits DESC""", limit=40)

In [ ]:
report("status_distribution", """
    SELECT status, COUNT(*) AS hits,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM access_logs), 3) AS pct
    FROM access_logs GROUP BY status ORDER BY hits DESC""")

report("errors_daily", """
    SELECT date, COUNT(*) AS total,
           SUM(CASE WHEN status = 404 THEN 1 ELSE 0 END) AS not_found,
           ROUND(100.0 * SUM(CASE WHEN status = 404 THEN 1 ELSE 0 END) / COUNT(*), 3) AS pct_404,
           ROUND(100.0 * SUM(CASE WHEN status >= 500 THEN 1 ELSE 0 END) / COUNT(*), 4) AS pct_5xx
    FROM access_logs GROUP BY date ORDER BY date""")

report("broken_links", """
    SELECT endpoint, COUNT(*) AS hits, COUNT(DISTINCT host) AS distinct_visitors
    FROM access_logs WHERE status = 404
    GROUP BY endpoint ORDER BY hits DESC""", limit=300)

In [ ]:
report("size_percentiles", """
    SELECT percentile_approx(bytes, 0.50) AS p50, percentile_approx(bytes, 0.90) AS p90,
           percentile_approx(bytes, 0.95) AS p95, percentile_approx(bytes, 0.99) AS p99,
           MAX(bytes) AS max_bytes, ROUND(AVG(bytes), 0) AS mean_bytes
    FROM access_logs WHERE status = 200 AND bytes > 0""")

report("size_buckets", """
    SELECT CASE WHEN bytes = 0 THEN '0 (no body)' WHEN bytes < 1024 THEN '<1 KB'
                WHEN bytes < 10240 THEN '1-10 KB' WHEN bytes < 102400 THEN '10-100 KB'
                WHEN bytes < 1048576 THEN '100 KB-1 MB' ELSE '>1 MB' END AS bucket,
           COUNT(*) AS hits
    FROM access_logs WHERE status = 200 GROUP BY bucket ORDER BY hits DESC""")

report("heaviest_hosts", """
    SELECT host, COUNT(*) AS hits, ROUND(SUM(bytes)/1024/1024, 1) AS mb
    FROM access_logs GROUP BY host ORDER BY hits DESC""", limit=300)

In [ ]:
spark.sql("""
    SELECT COUNT(*) AS requests, COUNT(DISTINCT host) AS unique_hosts,
           COUNT(DISTINCT date) AS days, ROUND(SUM(bytes)/1024/1024/1024, 2) AS gb_sent
    FROM access_logs""").show()

## 6. Algorithm — Z-score anomaly detection (Spark SQL, `jobs/anomalies.py`)
1. Count the requests in every (date, hour).
2. **Baseline** for each hour of the day: the mean and standard deviation across all days.
3. **z = (hits − mean) ÷ std**: how many standard deviations away from normal.
4. **|z| > 3** → anomaly (spike if z > 3, drop if z < −3).

In [ ]:
THRESHOLD = 3.0
anomaly_hours = spark.sql(f"""
    WITH hourly AS (
        SELECT date, hour, COUNT(*) AS hits FROM access_logs GROUP BY date, hour
    ),
    baseline AS (
        SELECT hour, AVG(hits) AS mean_hits, STDDEV_POP(hits) AS std_hits
        FROM hourly GROUP BY hour
    ),
    scored AS (
        SELECT h.date, h.hour, h.hits,
               ROUND(b.mean_hits, 1) AS normal_hits,
               ROUND(b.std_hits, 1)  AS std_hits,
               ROUND((h.hits - b.mean_hits) / b.std_hits, 2) AS z
        FROM hourly h JOIN baseline b ON h.hour = b.hour
    )
    SELECT date, hour, hits, normal_hits, std_hits, z,
           CASE WHEN z > 0 THEN 'spike' ELSE 'drop' END AS kind
    FROM scored WHERE ABS(z) > {THRESHOLD}
    ORDER BY ABS(z) DESC""")
anomaly_hours.write.mode("overwrite").parquet(f"{REPORTS}/anomaly_hours")
print("anomalous hours:", anomaly_hours.count())
anomaly_hours.show(20)

In [ ]:
anomaly_days = spark.sql(f"""
    WITH daily AS (SELECT date, COUNT(*) AS hits FROM access_logs GROUP BY date),
    baseline AS (SELECT AVG(hits) AS mean_hits, STDDEV_POP(hits) AS std_hits FROM daily)
    SELECT d.date, d.hits, ROUND(b.mean_hits, 1) AS normal_hits, ROUND(b.std_hits, 1) AS std_hits,
           ROUND((d.hits - b.mean_hits) / b.std_hits, 2) AS z,
           CASE WHEN d.hits > b.mean_hits THEN 'spike' ELSE 'drop' END AS kind
    FROM daily d CROSS JOIN baseline b
    WHERE ABS((d.hits - b.mean_hits) / b.std_hits) > {THRESHOLD}
    ORDER BY ABS((d.hits - b.mean_hits) / b.std_hits) DESC""")
anomaly_days.write.mode("overwrite").parquet(f"{REPORTS}/anomaly_days")
anomaly_days.show()

In [ ]:
# Before / after chart
import matplotlib.pyplot as plt, pandas as pd
hourly = spark.sql("""SELECT date, hour, COUNT(*) AS hits FROM access_logs
                      GROUP BY date, hour ORDER BY date, hour""").toPandas()
hourly["t"] = pd.to_datetime(hourly["date"].astype(str)) + pd.to_timedelta(hourly["hour"], unit="h")
an = anomaly_hours.toPandas()
an["t"] = pd.to_datetime(an["date"].astype(str)) + pd.to_timedelta(an["hour"], unit="h")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
a1.plot(hourly["t"], hourly["hits"], lw=0.8, color="#2a78d6")
a1.set_title("Before: requests per hour, 1 July - 31 August 1995"); a1.set_ylabel("requests / hour")
a2.plot(hourly["t"], hourly["hits"], lw=0.8, color="#2a78d6", label="requests per hour")
a2.scatter(an["t"], an["hits"], color="#e34948", s=40, zorder=5, label=f"anomaly |z| > 3 ({len(an)} hours)")
a2.set_title("After: Z-score anomaly detection"); a2.set_ylabel("requests / hour"); a2.legend(loc="upper right")
plt.tight_layout(); plt.show()

## 7. Everything is stored in HDFS: raw → curated → reports

In [ ]:
if HDFS_OK:
    sh("hdfs dfs -ls /data/weblogs")
    sh("hdfs dfs -ls /data/weblogs/reports")
    sh("hdfs dfs -du -s -h /data/weblogs/raw /data/weblogs/curated /data/weblogs/reports")
else:
    print("(HDFS not running: data is in", BASE, ")")

## 8. Machine learning: web-attack detection (Spark MLlib)
**Task:** supervised binary classification, **2 classes**: 0 = normal request, 1 = attack (SQL injection, XSS, command injection, path traversal).

The NASA logs have no labels, so the models learn from two labelled public datasets, **HttpParams** and **Fwaf**, also stored in HDFS.
**Our model:** Factorization Machine (`FMClassifier`), compared with 7 Spark MLlib models on the same features and test set.
Same code as `jobs/ml_preprocess.py` and `jobs/ml_train.py`. All 8 models take about 20 minutes on Colab; set `QUICK = True` to train only the top 3.

In [ ]:
QUICK = False   # True = only Factorization Machine, Linear SVM, Logistic Regression

# Download the labelled data (about 29 MB) and load it into HDFS: make ml-data + make ml-ingest
ADATA = f"{WORK}/data/attacks"
os.makedirs(ADATA, exist_ok=True)
for url, name in [("https://raw.githubusercontent.com/Morzeux/HttpParamsDataset/master/payload_full.csv", "payload_full.csv"),
                  ("https://raw.githubusercontent.com/faizann24/Fwaf-Machine-Learning-driven-Web-Application-Firewall/master/goodqueries.txt", "goodqueries.txt"),
                  ("https://raw.githubusercontent.com/faizann24/Fwaf-Machine-Learning-driven-Web-Application-Firewall/master/badqueries.txt", "badqueries.txt")]:
    if not os.path.exists(f"{ADATA}/{name}"):
        sh(f"curl -fsSL --retry 3 {url} -o {ADATA}/{name}", check=True)
if HDFS_OK:
    sh("hdfs dfs -mkdir -p /data/attacks")
    sh(f"hdfs dfs -put -f {ADATA}/* /data/attacks/")
    sh("hdfs dfs -du -h /data/attacks")
    ATTACK = "hdfs://localhost:8020/data/attacks"
else:
    ATTACK = f"file://{ADATA}"
print("ATTACK =", ATTACK)

### 8.1 ML preprocessing (`jobs/ml_preprocess.py`)
Label → URL-decode, lower-case, trim → **bias fix** (keep only the query part after `?`) → remove duplicates and conflicting labels → 80/20 split, training set balanced to 4 normal : 1 attack, test set keeps the real mix.

In [ ]:
from urllib.parse import unquote_plus
hp = (spark.read.option("header", True).option("multiLine", True).option("escape", '"')
      .csv(f"{ATTACK}/payload_full.csv")
      .select(F.col("payload").alias("text"), F.when(F.col("label") == "anom", 1).otherwise(0).alias("label"),
              F.lit("HttpParams").alias("source")))
good = spark.read.text(f"{ATTACK}/goodqueries.txt").select(F.col("value").alias("text"), F.lit(0).alias("label"), F.lit("Fwaf").alias("source"))
bad  = spark.read.text(f"{ATTACK}/badqueries.txt").select(F.col("value").alias("text"), F.lit(1).alias("label"), F.lit("Fwaf").alias("source"))
raw_ml = hp.unionByName(good).unionByName(bad)
print(f"loaded: {raw_ml.count():,} labelled requests")
raw_ml.groupBy("source", "label").count().orderBy("source", "label").show()

url_decode = F.udf(lambda t: unquote_plus(t) if t else t, "string")
clean_ml = (raw_ml.withColumn("text", F.lower(F.trim(url_decode("text")))).filter(F.length("text") > 0))
print("share of requests containing '?' BEFORE the bias fix:")
clean_ml.groupBy("label").agg(F.round(100 * F.avg(F.col("text").contains("?").cast("int")), 1).alias("pct_with_?")).orderBy("label").show()
clean_ml = clean_ml.withColumn("text", F.trim(F.when(F.instr("text", "?") > 0, F.expr("substring(text, instr(text, '?') + 1)"))
                                              .otherwise(F.col("text")))).filter(F.length("text") > 0)
ml_data = (clean_ml.groupBy("text").agg(F.max("label").alias("label"), F.min("label").alias("lo"))
           .filter("label = lo").drop("lo").cache())
print(f"after cleaning, de-duplication and removing conflicting labels: {ml_data.count():,} requests")
ml_data.groupBy("label").count().orderBy("label").show()

train_all, test_ml = ml_data.randomSplit([0.8, 0.2], seed=42)
n_att = train_all.filter("label = 1").count(); n_nor = train_all.filter("label = 0").count()
train_ml = train_all.sampleBy("label", {0: min(1.0, 4 * n_att / n_nor), 1: 1.0}, seed=42)
print(f"train (balanced 4:1): {train_ml.count():,}    test (real mix): {test_ml.count():,}")

### 8.2 Features and the 8 models (`jobs/ml_train.py`)
Each request becomes **12,292 features**: its characters, character pairs and triples (hashed into 3 × 4,096 buckets) plus its length and counts of special characters, digits and attack keywords.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import NGram, HashingTF, VectorAssembler, MaxAbsScaler
from pyspark.ml.classification import (FMClassifier, LogisticRegression, LinearSVC, NaiveBayes, DecisionTreeClassifier,
                                       RandomForestClassifier, GBTClassifier, MultilayerPerceptronClassifier)
SPECIAL = r"""['"<>;()=%&|`$\-\.\\/*+#!{}\[\]]"""
def add_text_features(df):
    return (df.withColumn("chars", F.expr("filter(split(text, ''), c -> c != '')"))
        .withColumn("len", F.length("text").cast("double"))
        .withColumn("n_special", (F.length("text") - F.length(F.regexp_replace("text", SPECIAL, ""))).cast("double"))
        .withColumn("n_digits", (F.length("text") - F.length(F.regexp_replace("text", "[0-9]", ""))).cast("double"))
        .withColumn("n_keywords", F.lit(0.0)
            + F.col("text").rlike(r"select|union|insert|drop|update|delete|sleep\(|benchmark").cast("double")
            + F.col("text").rlike(r"<script|javascript:|onerror|onload|alert\(").cast("double")
            + F.col("text").rlike(r"\.\./|/etc/passwd|win\.ini|boot\.ini").cast("double")
            + F.col("text").rlike(r";\s*(ls|cat|id|uname|wget|curl)|\|\s*(ls|cat|id)|`").cast("double")))
def feature_stages(n, out):
    return [HashingTF(inputCol="chars", outputCol=f"{out}_1", numFeatures=n), HashingTF(inputCol="bi", outputCol=f"{out}_2", numFeatures=n),
            HashingTF(inputCol="tri", outputCol=f"{out}_3", numFeatures=n),
            VectorAssembler(inputCols=[f"{out}_1", f"{out}_2", f"{out}_3", "len", "n_special", "n_digits", "n_keywords"], outputCol=f"{out}_raw"),
            MaxAbsScaler(inputCol=f"{out}_raw", outputCol=out)]
featurizer = Pipeline(stages=[NGram(n=2, inputCol="chars", outputCol="bi"), NGram(n=3, inputCol="chars", outputCol="tri")]
                      + feature_stages(4096, "features") + feature_stages(128, "features_small"))   # MLP needs a small, dense input
feat_model = featurizer.fit(add_text_features(train_ml))
tr = feat_model.transform(add_text_features(train_ml)).select("label", "features", "features_small").cache()
te = feat_model.transform(add_text_features(test_ml)).select("label", "features", "features_small").cache()
print(f"train {tr.count():,}  test {te.count():,}")

models = {
    "Factorization Machine (ours)": FMClassifier(factorSize=8, maxIter=100, stepSize=0.2, regParam=1e-4, seed=42),
    "Linear SVM":             LinearSVC(maxIter=100, regParam=1e-4),
    "Logistic Regression":    LogisticRegression(maxIter=100, regParam=1e-4),
    "Naive Bayes":            NaiveBayes(modelType="multinomial"),
    "Decision Tree":          DecisionTreeClassifier(maxDepth=10, seed=42),
    "Random Forest":          RandomForestClassifier(numTrees=30, maxDepth=10, seed=42),
    "Gradient Boosted Trees": GBTClassifier(maxIter=15, maxDepth=5, subsamplingRate=0.5, featureSubsetStrategy="0.3", seed=42),
    "Multilayer Perceptron":  MultilayerPerceptronClassifier(layers=[3 * 128 + 4, 32, 2], maxIter=100, featuresCol="features_small", seed=42),
}
if QUICK:
    models = {k: v for k, v in list(models.items())[:3]}

def evaluate(pred):
    c = {(int(r["label"]), int(r["prediction"])): r["count"] for r in pred.groupBy("label", "prediction").count().collect()}
    tn, fp, fn, tp = c.get((0, 0), 0), c.get((0, 1), 0), c.get((1, 0), 0), c.get((1, 1), 0)
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    return dict(tn=tn, fp=fp, fn=fn, tp=tp, accuracy=round((tp + tn) / (tp + tn + fp + fn), 4),
                precision=round(p, 4), recall=round(r, 4), f1=round(2 * p * r / max(p + r, 1e-12), 4))

results, fitted = [], {}
for name, est in models.items():
    t0 = time.time(); m = est.fit(tr); secs = round(time.time() - t0, 1)
    r = {"model": name, "train_seconds": secs, **evaluate(m.transform(te))}
    results.append(r); fitted[name] = m
    print(f"{name:<30} F1 {r['f1']:.4f}  precision {r['precision']:.4f}  recall {r['recall']:.4f}  ({secs}s)")
results.sort(key=lambda r: -r["f1"])

### 8.3 Results: comparison and confusion matrices

In [ ]:
import pandas as pd, numpy as np
display(pd.DataFrame(results)[["model", "accuracy", "precision", "recall", "f1", "tn", "fp", "fn", "tp", "train_seconds"]])

cols = 4; rows = (len(results) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.9 * rows))
for ax in np.atleast_1d(axes).flat: ax.axis("off")
for ax, r in zip(np.atleast_1d(axes).flat, results):
    ax.axis("on"); m = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]]); s = m / m.sum(axis=1, keepdims=True)
    ax.imshow(s, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{m[i,j]:,}\n{100*s[i,j]:.1f}%", ha="center", va="center", color="white" if s[i,j] > .5 else "black", fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(["Normal", "Attack"]); ax.set_yticklabels(["Normal", "Attack"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"{r['model']}\nF1 = {r['f1']:.4f}", fontsize=10)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 0.5 * len(results) + 1))
ax.barh([r["model"] for r in results][::-1], [r["f1"] for r in results][::-1],
        color=["#2a78d6" if "ours" in r["model"] else "#9aa4b2" for r in results][::-1])
for i, r in enumerate(results[::-1]): ax.text(r["f1"] + 0.002, i, f"{r['f1']:.4f}", va="center")
ax.set_xlim(min(r["f1"] for r in results) - 0.05, 1.0); ax.set_title("F1 on the test set"); plt.tight_layout(); plt.show()

### 8.4 The Factorization Machine on the NASA logs
The trained model labels every distinct NASA URL. There are no labels to check against, so the flagged URLs are **candidates**: read them to judge how many are real attacks.

In [ ]:
nasa_urls = (curated.groupBy("endpoint").agg(F.count("*").alias("hits"), F.countDistinct("host").alias("hosts"))
             .withColumn("text", F.lower(F.trim(url_decode("endpoint"))))
             .withColumn("text", F.trim(F.when(F.instr("text", "?") > 0, F.expr("substring(text, instr(text, '?') + 1)")).otherwise(F.col("text"))))
             .filter(F.length("text") > 0))
scored = (fitted["Factorization Machine (ours)"].transform(feat_model.transform(add_text_features(nasa_urls)))
          .withColumn("p_attack", F.round(vector_to_array("probability")[1], 3)))
flagged = scored.filter("prediction = 1").select("endpoint", "hits", "hosts", "p_attack")
print(f"{scored.count():,} distinct NASA URLs scored, {flagged.count():,} flagged as attack")
flagged.orderBy(F.desc("p_attack"), F.desc("hits")).show(25, truncate=80)

**Summary:** Kriti stored the raw logs in **Hadoop HDFS**; Kushal's **Spark** ETL turned 3.46 million text lines into a clean 11-column table (Parquet, split by date); Gargi's **Spark SQL** produced the 11 reports and found the unusual hours with **Z-score anomaly detection**; **Spark MLlib** trained our Factorization Machine and 7 comparison models to detect web attacks (2 classes), with a confusion matrix for each.